# 🎨 Laboratorio: Conditional DCGAN con Fashion-MNIST

## 🎓 Diplomado en Inteligencia Artificial - UTFSM
### 📚 Curso: IA Generativa

---

**Instructor:** Pablo Álvarez  
**Web:** [Multivérsica](https://multiversica.com/)  
**Fecha:** Diciembre 2025

---

## 🎯 Objetivo del Laboratorio

En este ejercicio implementaremos una **Conditional DCGAN** (Deep Convolutional GAN) utilizando el dataset **Fashion-MNIST**.

A diferencia de una GAN tradicional que genera imágenes aleatorias del espacio latente, una **cGAN** recibe una **etiqueta (condición)** tanto en el Generador como en el Discriminador. Esto nos permitirá **controlar la salida**: podremos decirle a la red:

> *"Genera una zapatilla"* o *"Genera un abrigo"*

---

## 🧠 Conceptos Clave

| Concepto | Descripción |
|----------|-------------|
| **Juego Minimax** | Competencia entre Generador y Discriminador |
| **Condicionamiento** | Uso de Embeddings para inyectar información de clase |
| **Espacio Latente** | Vector de ruido que se transforma en imágenes |
| **Interpolación** | Transición suave entre clases en el espacio latente |

---

## 📊 Arquitectura cGAN

```
┌─────────────────────────────────────────────────────────────────┐
│                     CONDITIONAL DCGAN                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Ruido z ──┐                                                   │
│             ├──► [GENERADOR] ──► Imagen Falsa ──┐               │
│   Label y ──┘                                   │               │
│                                                 ├──► [DISCRIMINADOR] ──► Real/Falso
│   Imagen Real ──────────────────────────────────┘               │
│   Label y ──────────────────────────────────────────────────────┘
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 📦 Parte 1: Configuración del Entorno

Importamos las librerías necesarias y configuramos el dispositivo GPU.

In [ ]:
# ============================================================
# 📦 IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuración del dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## ⚙️ Parte 2: Hiperparámetros

Definimos los hiperparámetros de entrenamiento.

In [ ]:
# ============================================================
# ⚙️ HIPERPARÁMETROS
# ============================================================

# Dimensiones
LATENT_DIM = 100      # Dimensión del espacio latente (ruido z)
IMG_SIZE = 28         # Tamaño de imagen Fashion-MNIST
CHANNELS = 1          # Imágenes en escala de grises
NUM_CLASSES = 10      # 10 categorías de ropa
EMBED_DIM = 50        # Dimensión del embedding de clase

# Entrenamiento
BATCH_SIZE = 128
EPOCHS = 50
LR_G = 0.0002         # Learning rate del Generador
LR_D = 0.0002         # Learning rate del Discriminador
BETA1 = 0.5           # Adam beta1
BETA2 = 0.999         # Adam beta2

# Clases de Fashion-MNIST
CLASS_NAMES = [
    'Camiseta/Top', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
    'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Bota'
]

print("✅ Hiperparámetros configurados")
print(f"📐 Espacio latente: {LATENT_DIM} dimensiones")
print(f"🏷️ Clases: {NUM_CLASSES}")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"🔄 Epochs: {EPOCHS}")

## 📊 Parte 3: Carga del Dataset Fashion-MNIST

Fashion-MNIST contiene 70,000 imágenes de 10 categorías de ropa.

In [ ]:
# ============================================================
# 📊 CARGA DEL DATASET
# ============================================================

# Transformaciones
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # Normalizar a [-1, 1]
])

# Descargar Fashion-MNIST
train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f"✅ Dataset cargado: {len(train_dataset)} imágenes")
print(f"📦 Batches por epoch: {len(train_loader)}")

In [ ]:
# ============================================================
# 👁️ VISUALIZACIÓN DEL DATASET
# ============================================================

def show_samples(loader, n_samples=10):
    """Muestra ejemplos del dataset con sus etiquetas."""
    images, labels = next(iter(loader))
    
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle('🎽 Ejemplos de Fashion-MNIST', fontsize=14, fontweight='bold')
    
    for i, ax in enumerate(axes.flat):
        img = images[i].squeeze().numpy()
        img = (img + 1) / 2  # Desnormalizar
        ax.imshow(img, cmap='gray')
        ax.set_title(CLASS_NAMES[labels[i]], fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

show_samples(train_loader)

## 🏗️ Parte 4: Arquitectura del Generador

El **Generador** recibe:
- Vector de ruido `z` (100 dimensiones)
- Etiqueta de clase `y` (embedding de 50 dimensiones)

Y produce una imagen de 28x28 píxeles.

In [ ]:
# ============================================================
# 🎨 GENERADOR CONDICIONAL
# ============================================================

class Generator(nn.Module):
    """
    Generador Condicional para cGAN.
    
    Recibe:
        - z: Vector de ruido (LATENT_DIM,)
        - labels: Etiquetas de clase (batch_size,)
    
    Retorna:
        - Imágenes generadas (batch_size, 1, 28, 28)
    """
    
    def __init__(self, latent_dim, num_classes, embed_dim, img_channels):
        super(Generator, self).__init__()
        
        # Embedding para las etiquetas de clase
        self.label_embedding = nn.Embedding(num_classes, embed_dim)
        
        # Dimensión de entrada: ruido + embedding
        input_dim = latent_dim + embed_dim
        
        # Red generadora
        self.model = nn.Sequential(
            # Capa 1: (input_dim) -> (256, 7, 7)
            nn.Linear(input_dim, 256 * 7 * 7),
            nn.BatchNorm1d(256 * 7 * 7),
            nn.LeakyReLU(0.2),
            
            # Reshape a tensor 3D
            nn.Unflatten(1, (256, 7, 7)),
            
            # Capa 2: Upsampling (256, 7, 7) -> (128, 14, 14)
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            
            # Capa 3: Upsampling (128, 14, 14) -> (64, 28, 28)
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            
            # Capa final: (64, 28, 28) -> (1, 28, 28)
            nn.Conv2d(64, img_channels, kernel_size=3, stride=1, padding=1),
            nn.Tanh()  # Salida en [-1, 1]
        )
    
    def forward(self, z, labels):
        # Obtener embedding de la etiqueta
        label_embed = self.label_embedding(labels)
        
        # Concatenar ruido + embedding
        x = torch.cat([z, label_embed], dim=1)
        
        # Generar imagen
        return self.model(x)

# Crear generador
generator = Generator(LATENT_DIM, NUM_CLASSES, EMBED_DIM, CHANNELS).to(device)
print("✅ Generador creado")
print(f"📊 Parámetros: {sum(p.numel() for p in generator.parameters()):,}")

## 🔍 Parte 5: Arquitectura del Discriminador

El **Discriminador** recibe:
- Imagen (real o generada) de 28x28
- Etiqueta de clase `y`

Y clasifica si la imagen es **real** o **falsa**.

In [ ]:
# ============================================================
# 🔍 DISCRIMINADOR CONDICIONAL
# ============================================================

class Discriminator(nn.Module):
    """
    Discriminador Condicional para cGAN.
    
    Recibe:
        - images: Imágenes (batch_size, 1, 28, 28)
        - labels: Etiquetas de clase (batch_size,)
    
    Retorna:
        - Probabilidad de que sea real (batch_size, 1)
    """
    
    def __init__(self, num_classes, img_channels, img_size):
        super(Discriminator, self).__init__()
        
        # Embedding para las etiquetas (proyectado al tamaño de imagen)
        self.label_embedding = nn.Embedding(num_classes, img_size * img_size)
        self.img_size = img_size
        
        # Red discriminadora
        # Entrada: imagen (1 canal) + embedding (1 canal) = 2 canales
        self.model = nn.Sequential(
            # Capa 1: (2, 28, 28) -> (64, 14, 14)
            nn.Conv2d(img_channels + 1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(0.3),
            
            # Capa 2: (64, 14, 14) -> (128, 7, 7)
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(0.3),
            
            # Capa 3: (128, 7, 7) -> (256, 3, 3)
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(0.3),
            
            # Aplanar y clasificar
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 1),
            nn.Sigmoid()  # Probabilidad [0, 1]
        )
    
    def forward(self, images, labels):
        # Obtener embedding y reshape a imagen
        label_embed = self.label_embedding(labels)
        label_embed = label_embed.view(-1, 1, self.img_size, self.img_size)
        
        # Concatenar imagen + embedding como canal adicional
        x = torch.cat([images, label_embed], dim=1)
        
        # Clasificar
        return self.model(x)

# Crear discriminador
discriminator = Discriminator(NUM_CLASSES, CHANNELS, IMG_SIZE).to(device)
print("✅ Discriminador creado")
print(f"📊 Parámetros: {sum(p.numel() for p in discriminator.parameters()):,}")

## ⚡ Parte 6: Configuración del Entrenamiento

Configuramos los optimizadores y la función de pérdida.

In [ ]:
# ============================================================
# ⚡ CONFIGURACIÓN DE ENTRENAMIENTO
# ============================================================

# Función de pérdida: Binary Cross Entropy
criterion = nn.BCELoss()

# Optimizadores Adam
optimizer_G = optim.Adam(generator.parameters(), lr=LR_G, betas=(BETA1, BETA2))
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR_D, betas=(BETA1, BETA2))

# Etiquetas fijas para visualización durante entrenamiento
fixed_noise = torch.randn(100, LATENT_DIM, device=device)
fixed_labels = torch.tensor([i for i in range(10) for _ in range(10)], device=device)

print("✅ Entrenamiento configurado")
print(f"📉 Función de pérdida: Binary Cross Entropy")
print(f"🔧 Optimizador: Adam (lr={LR_G}, betas=({BETA1}, {BETA2}))")

In [ ]:
# ============================================================
# 👁️ FUNCIÓN DE VISUALIZACIÓN
# ============================================================

def show_generated_images(generator, noise, labels, epoch=None):
    """Visualiza imágenes generadas organizadas por clase."""
    generator.eval()
    with torch.no_grad():
        fake_images = generator(noise, labels).cpu()
    generator.train()
    
    # Desnormalizar
    fake_images = (fake_images + 1) / 2
    
    fig, axes = plt.subplots(10, 10, figsize=(12, 12))
    title = '🎨 Imágenes Generadas por Clase'
    if epoch is not None:
        title += f' (Epoch {epoch})'
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for i, ax in enumerate(axes.flat):
        img = fake_images[i].squeeze().numpy()
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    
    # Añadir nombres de clase en el lado izquierdo
    for i, ax in enumerate(axes[:, 0]):
        ax.set_ylabel(CLASS_NAMES[i], fontsize=8, rotation=0, ha='right')
    
    plt.tight_layout()
    plt.show()

## 🔄 Parte 7: Loop de Entrenamiento

Implementamos el **juego minimax** entre Generador y Discriminador:

1. **Entrenar Discriminador**: Maximizar `log(D(x|y)) + log(1 - D(G(z|y)|y))`
2. **Entrenar Generador**: Minimizar `log(1 - D(G(z|y)|y))` (o maximizar `log(D(G(z|y)|y))`)

In [ ]:
# ============================================================
# 🔄 LOOP DE ENTRENAMIENTO
# ============================================================

# Listas para guardar pérdidas
G_losses = []
D_losses = []

print("🚀 Iniciando entrenamiento...")
print("="*60)

for epoch in range(EPOCHS):
    g_loss_epoch = 0
    d_loss_epoch = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    
    for batch_idx, (real_images, labels) in enumerate(pbar):
        batch_size = real_images.size(0)
        real_images = real_images.to(device)
        labels = labels.to(device)
        
        # Etiquetas para real (1) y fake (0)
        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)
        
        # ============================
        # 1. ENTRENAR DISCRIMINADOR
        # ============================
        optimizer_D.zero_grad()
        
        # Pérdida con imágenes reales
        output_real = discriminator(real_images, labels)
        loss_real = criterion(output_real, real_labels)
        
        # Generar imágenes falsas
        noise = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_images = generator(noise, labels)
        
        # Pérdida con imágenes falsas
        output_fake = discriminator(fake_images.detach(), labels)
        loss_fake = criterion(output_fake, fake_labels)
        
        # Pérdida total del discriminador
        loss_D = (loss_real + loss_fake) / 2
        loss_D.backward()
        optimizer_D.step()
        
        # ============================
        # 2. ENTRENAR GENERADOR
        # ============================
        optimizer_G.zero_grad()
        
        # El generador quiere que el discriminador clasifique las falsas como reales
        output_fake = discriminator(fake_images, labels)
        loss_G = criterion(output_fake, real_labels)
        
        loss_G.backward()
        optimizer_G.step()
        
        # Acumular pérdidas
        g_loss_epoch += loss_G.item()
        d_loss_epoch += loss_D.item()
        
        # Actualizar barra de progreso
        pbar.set_postfix({
            'D_loss': f'{loss_D.item():.4f}',
            'G_loss': f'{loss_G.item():.4f}'
        })
    
    # Promediar pérdidas del epoch
    G_losses.append(g_loss_epoch / len(train_loader))
    D_losses.append(d_loss_epoch / len(train_loader))
    
    # Mostrar imágenes cada 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"\n📊 Epoch {epoch+1}: D_loss={D_losses[-1]:.4f}, G_loss={G_losses[-1]:.4f}")
        show_generated_images(generator, fixed_noise, fixed_labels, epoch+1)

print("\n" + "="*60)
print("✅ Entrenamiento completado!")

## 📈 Parte 8: Visualización de Resultados

In [ ]:
# ============================================================
# 📈 GRÁFICA DE PÉRDIDAS
# ============================================================

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(G_losses, label='Generador', color='blue', linewidth=2)
plt.plot(D_losses, label='Discriminador', color='orange', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Pérdida')
plt.title('📉 Evolución de Pérdidas')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(G_losses, label='Generador', color='blue', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Pérdida del Generador')
plt.title('🎨 Pérdida del Generador')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 🎨 GENERACIÓN FINAL POR CLASE
# ============================================================

print("🎨 Generando imágenes finales por clase...")
show_generated_images(generator, fixed_noise, fixed_labels, epoch=EPOCHS)

## 🎯 Parte 9: Generación Controlada

Ahora podemos generar prendas específicas usando el condicionamiento.

In [ ]:
# ============================================================
# 🎯 GENERACIÓN CONTROLADA POR CLASE
# ============================================================

def generate_specific_class(generator, class_idx, n_samples=10):
    """Genera múltiples muestras de una clase específica."""
    generator.eval()
    
    noise = torch.randn(n_samples, LATENT_DIM, device=device)
    labels = torch.full((n_samples,), class_idx, device=device, dtype=torch.long)
    
    with torch.no_grad():
        fake_images = generator(noise, labels).cpu()
    
    # Desnormalizar
    fake_images = (fake_images + 1) / 2
    
    fig, axes = plt.subplots(1, n_samples, figsize=(15, 2))
    fig.suptitle(f'🏷️ Generando: {CLASS_NAMES[class_idx]}', fontsize=14, fontweight='bold')
    
    for i, ax in enumerate(axes):
        img = fake_images[i].squeeze().numpy()
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Generar diferentes clases
print("🎯 GENERACIÓN CONTROLADA")
print("="*60)

for class_idx in [7, 4, 1, 8]:  # Zapatilla, Abrigo, Pantalón, Bolso
    generate_specific_class(generator, class_idx)

## 🌈 Parte 10: Interpolación en el Espacio Latente

Exploramos transiciones suaves entre dos puntos del espacio latente.

In [ ]:
# ============================================================
# 🌈 INTERPOLACIÓN EN EL ESPACIO LATENTE
# ============================================================

def interpolate_latent(generator, class_idx, n_steps=10):
    """Interpola entre dos puntos aleatorios del espacio latente."""
    generator.eval()
    
    # Dos puntos aleatorios en el espacio latente
    z1 = torch.randn(1, LATENT_DIM, device=device)
    z2 = torch.randn(1, LATENT_DIM, device=device)
    
    # Interpolación lineal
    alphas = torch.linspace(0, 1, n_steps)
    labels = torch.full((1,), class_idx, device=device, dtype=torch.long)
    
    images = []
    with torch.no_grad():
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            img = generator(z, labels).cpu()
            images.append(img)
    
    images = torch.cat(images, dim=0)
    images = (images + 1) / 2
    
    fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))
    fig.suptitle(f'🌈 Interpolación: {CLASS_NAMES[class_idx]}', fontsize=14, fontweight='bold')
    
    for i, ax in enumerate(axes):
        img = images[i].squeeze().numpy()
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if i == 0:
            ax.set_title('z₁', fontsize=10)
        elif i == n_steps - 1:
            ax.set_title('z₂', fontsize=10)
    
    plt.tight_layout()
    plt.show()

print("🌈 INTERPOLACIÓN EN ESPACIO LATENTE")
print("="*60)

for class_idx in [7, 0, 4]:  # Zapatilla, Camiseta, Abrigo
    interpolate_latent(generator, class_idx)

In [ ]:
# ============================================================
# 🔄 INTERPOLACIÓN ENTRE CLASES
# ============================================================

def interpolate_classes(generator, class1, class2, n_steps=10):
    """Interpola entre dos clases con el mismo ruido."""
    generator.eval()
    
    # Mismo ruido para ambas
    z = torch.randn(1, LATENT_DIM, device=device)
    
    # Embeddings de las dos clases
    embed1 = generator.label_embedding(torch.tensor([class1], device=device))
    embed2 = generator.label_embedding(torch.tensor([class2], device=device))
    
    alphas = torch.linspace(0, 1, n_steps)
    
    images = []
    with torch.no_grad():
        for alpha in alphas:
            # Interpolar embeddings
            embed = (1 - alpha) * embed1 + alpha * embed2
            x = torch.cat([z, embed], dim=1)
            img = generator.model(x).cpu()
            images.append(img)
    
    images = torch.cat(images, dim=0)
    images = (images + 1) / 2
    
    fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))
    fig.suptitle(f'🔄 {CLASS_NAMES[class1]} → {CLASS_NAMES[class2]}', fontsize=14, fontweight='bold')
    
    for i, ax in enumerate(axes):
        img = images[i].squeeze().numpy()
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("🔄 INTERPOLACIÓN ENTRE CLASES")
print("="*60)

# Transiciones interesantes
interpolate_classes(generator, 7, 9)  # Zapatilla -> Bota
interpolate_classes(generator, 0, 6)  # Camiseta -> Camisa
interpolate_classes(generator, 3, 4)  # Vestido -> Abrigo

## 💾 Parte 11: Guardar y Cargar Modelo

In [ ]:
# ============================================================
# 💾 GUARDAR MODELOS
# ============================================================

# Guardar pesos
torch.save({
    'generator': generator.state_dict(),
    'discriminator': discriminator.state_dict(),
    'optimizer_G': optimizer_G.state_dict(),
    'optimizer_D': optimizer_D.state_dict(),
    'epoch': EPOCHS,
    'G_losses': G_losses,
    'D_losses': D_losses
}, 'cgan_fashion_mnist.pth')

print("✅ Modelo guardado: cgan_fashion_mnist.pth")

---

## 📝 Ejercicios Propuestos

### 🎯 Ejercicio 1: Experimentar con Hiperparámetros
Modifica los siguientes hiperparámetros y observa el impacto:
- `LATENT_DIM`: Prueba con 50, 150, 200
- `EMBED_DIM`: Prueba con 25, 100
- `LR_G` y `LR_D`: Prueba diferentes ratios (ej: `LR_D = 0.0001`)

### 🎯 Ejercicio 2: Mejorar la Arquitectura
- Añade más capas convolucionales al Generador
- Implementa **Spectral Normalization** en el Discriminador
- Prueba con **Instance Normalization** en lugar de BatchNorm

### 🎯 Ejercicio 3: Nuevas Funciones de Pérdida
Implementa:
- **Wasserstein Loss** (WGAN)
- **Hinge Loss** (SAGAN)
- **Least Squares Loss** (LSGAN)

### 🎯 Ejercicio 4: Otros Datasets
Adapta el código para:
- **MNIST** (dígitos escritos a mano)
- **CIFAR-10** (imágenes a color 32x32)
- **CelebA** (rostros de celebridades)

### 🎯 Ejercicio 5: Métricas de Evaluación
Implementa:
- **FID (Fréchet Inception Distance)**
- **IS (Inception Score)**
- **Mode Coverage**

---

## 🎓 Conclusiones

### ✅ Lo que aprendimos:

1. **Arquitectura cGAN**: Cómo el condicionamiento permite control sobre la generación
2. **Embeddings**: Representación de clases en espacio continuo
3. **Juego Minimax**: Balance entre Generador y Discriminador
4. **Espacio Latente**: Propiedades de interpolación y transición

### 📊 Métricas Clave:

| Métrica | Valor Esperado |
|---------|----------------|
| D_loss final | ~0.5-0.7 |
| G_loss final | ~0.8-1.5 |
| Calidad visual | Buena definición de formas |
| Control de clase | Alta correlación |

### 🚀 Próximos Pasos:

- Explorar **Progressive Growing GANs**
- Implementar **StyleGAN** para mejor calidad
- Estudiar **Diffusion Models** como alternativa moderna

---

## 📚 Referencias

1. **Conditional GANs**: Mirza, M., & Osindero, S. (2014). *Conditional Generative Adversarial Nets*
2. **DCGANs**: Radford, A., Metz, L., & Chintala, S. (2015). *Unsupervised Representation Learning with DCGANs*
3. **Fashion-MNIST**: Xiao, H., Rasul, K., & Vollgraf, R. (2017). *Fashion-MNIST: a Novel Image Dataset*

---

### 🎓 Diplomado en Inteligencia Artificial - UTFSM
### 📚 Curso: IA Generativa

**Instructor:** Pablo Álvarez  
**Web:** [Multivérsica](https://multiversica.com/)  

---

**¡Felicitaciones por completar el laboratorio! 🎉🇨🇱**